### **Importing Libraries** <a id="head1"></a>

In [ ]:
import numpy as np
import pandas as pd
                                                                                   
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve,roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

import pickle

import warnings
warnings.filterwarnings('ignore')

### **Loading Data** <a id="head2"></a>

In [ ]:
telco_customer = pd.read_csv("ChurnTrainDataset.csv")

In [ ]:
telco_customer

### **Preprocessing** <a id="head5"></a>

In [ ]:
# Encoding categorical data using cat codes
for col in telco_customer.columns[telco_customer.dtypes == 'object']:
    if col!='churn':
        telco_customer[col]=telco_customer[col].astype('category').cat.codes

# Fill Null Values of target column
telco_customer['churn'] =  telco_customer['churn'].fillna(telco_customer['churn'].mode()[0])

# Manual encoding the target variable
a={'yes':1,'no':0}
telco_customer['churn']=telco_customer['churn'].map(a)

### **Seperate Features & Target Variable** <a id="head6"></a>

In [ ]:
X = telco_customer.drop('churn',axis=1)
y=telco_customer['churn']

In [ ]:
cat_cols = X.select_dtypes(include=['int8']).columns.values

In [ ]:
num_cols = X.select_dtypes(include=['float64']).columns.values

### **Train Test Split**  <a id="head7"></a>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)

## Feature Engineering pipeline

In [ ]:
# Imputation Transformer to fill null values
fill_null_col = ColumnTransformer([
        ('FillCat',SimpleImputer(strategy='most_frequent'),cat_cols),
        ('FillNumeric',SimpleImputer(strategy='median'),num_cols),
    ],remainder='passthrough')

# Scaling
scale_col = ColumnTransformer([
    ('scale', StandardScaler(),slice(5,18))
])

# Principal Component Analysis
pca_col = ColumnTransformer([
    ('PCA', PCA(n_components=10),slice(0,18))
])

# # Model
logistic=LogisticRegression()
decision=DecisionTreeClassifier()
random = RandomForestClassifier()

### **Model Pipeline** <a id="head9"></a>

In [ ]:
pipeline_model1 = Pipeline([('fill_null_col', fill_null_col),
                 ('scale_col', scale_col),
                 ('pca_col', pca_col),
                 ('classifier1', logistic)])

pipeline_model2 = Pipeline([('fill_null_col', fill_null_col),
                 ('scale_col', scale_col),
                 ('pca_col', pca_col),
                 ('classifier2', decision)])

pipeline_model3 = Pipeline([('fill_null_col', fill_null_col),
                 ('scale_col', scale_col),
                 ('pca_col', pca_col),
                 ('classifier3', random)])

In [ ]:
pipelines=[pipeline_model1,pipeline_model2,pipeline_model3]

In [ ]:
best_accuracy=0.0
best_classifier=0
best_pipeline=""

In [ ]:
pipe_dict={0:'Logistic Regression',1:'Decision Tree',2:'Random Forest'}

# Fitting pipelines
for pipe in pipelines:
    pipe.fit(X_train,y_train)

In [ ]:
for i,model in enumerate(pipelines):
    print("{} Test Accuracy : {}".format(pipe_dict[i],model.score(X_test,y_test)))

In [ ]:
for i,model in enumerate(pipelines):
    pred=model.predict(X_test)
    print("\n{} Classification Report :\n {}".format(pipe_dict[i],classification_report(y_test,pred)))

In [ ]:
for i,model in enumerate(pipelines):
    if model.score(X_test,y_test)>best_accuracy:
        best_accuracy=model.score(X_test,y_test)
        best_pipeline=model
        best_classifier=i
print('Classifier with the best accuracy:{}'.format(pipe_dict[best_classifier]))

### **Best Hyperparameters for Random Forest Classifier Using GridSearchCV using a pipeline** 
<a id="head10"></a>

In [ ]:
# params = [{"classifier3__n_estimators": range(100, 501, 100),
#            "classifier3__max_depth":range(10,30,10),
#            "classifier3__min_samples_leaf":[1, 2, 4],
#            "classifier3__min_samples_split": [2, 5, 10],
#           }]
                 
# randomized_cv = GridSearchCV(estimator = pipeline_model3,
#                            param_grid = params,
#                            cv = 10 ,verbose = 1, n_jobs=-1)

# randomized_cv.fit(X_train,y_train)
# randomized_cv.best_params_

In [ ]:
randomized_cv = Pipeline([('fill_null_col', fill_null_col),
                 ('scale_col', scale_col),
                 ('pca_col', pca_col),
                 ('classifier3', RandomForestClassifier(max_depth=20,min_samples_leaf=1,min_samples_split=5,n_estimators=400))])

randomized_cv.fit(X_train,y_train)
filename = 'modelv1.pkl'
pickle.dump(randomized_cv, open(filename, 'wb'))

### Load the model from disk

In [ ]:
filename = 'modelv1.pkl'
loaded_model = pickle.load(open(filename, 'rb'))
result = loaded_model.score(X_test, y_test)
print(result)

### **Model Prediction**<a id="head11"></a>

In [ ]:
new = loaded_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_test, new)

In [ ]:
thresholds

In [ ]:
precision[:-1].shape

In [ ]:
recall[:-1].shape

In [ ]:
thresholds.shape

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
plt.plot(thresholds, precision[:-1], c ='r', label ='PRECISION') 

plt.plot(thresholds, recall[:-1], c ='b', label ='RECALL') 
plt.grid() 
plt.legend() 

plt.title('Precision-Recall Curve')

In [ ]:
fscore = (2 * precision * recall) / (precision + recall)

In [ ]:
index = np.argmax(fscore)
thresholdOpt = round(thresholds[index], ndigits = 4)
fscoreOpt = round(fscore[index], ndigits = 4)

In [ ]:
print('Best Threshold: {} with F-Score: {}'.format(thresholdOpt, fscoreOpt))

In [ ]:
threshold = thresholdOpt

In [ ]:
test = X_test.sample(10)
test

In [ ]:
print(loaded_model.predict_proba(test))
print(loaded_model.predict(test))

In [ ]:
y_pred_proba_new_threshold = (loaded_model.predict_proba(test)[:,1] >= threshold).astype(int)
y_pred_proba_new_threshold

In [ ]:
# p(a) = 0.7
# p(b) = 0.3
# if p(a)>=p(b):
#     then class a
# else class b

In [ ]:
# if p(1)>=0.43:
#     then class 1
# else class 0

In [ ]:
# p(1) = 0.45
# p(0) = 0.55